# Inspect MAE Checkpoints
Load saved MAE checkpoints and compare reconstruction quality (MSE + SSIM) on the same images.

In [ ]:
# --- Section 0: Imports and Paths ---
from pathlib import Path
import json

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import transforms as T

from data.animals10 import IMAGENET_MEAN, IMAGENET_STD, build_dataloaders
from training.mae_trainer import load_mae_model, reconstruct_mae_images
from training.unet import apply_patch_mask
from utils.common import tensor_to_numpy_image

CKPT_DIR = Path("/kaggle/working/checkpoints_mae_only")
RESULT_DIR = Path("/kaggle/working/results_mae_only")
COMPARE_DIR = RESULT_DIR / "checkpoint_comparison"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT = "/kaggle/input/datasets/alessiocorrado99/animals10/raw-img"
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
MASK_RATIO = 0.75

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Checkpoint dir:", CKPT_DIR)
print("Comparison dir:", COMPARE_DIR)

In [ ]:
# --- Section 1: Build Validation Loader and Fixed Samples ---
val_transform = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# train_transform is required by build_dataloaders API, but unused for this notebook
dummy_train_transform = val_transform

_, val_loader, _ = build_dataloaders(
    root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    val_fraction=0.2,
    seed=42,
    num_workers=NUM_WORKERS,
    train_transform=dummy_train_transform,
    val_transform=val_transform,
    use_weighted_sampler=False,
)

fixed_batch = next(iter(val_loader))
fixed_images = fixed_batch[0][:4].to(device)
print("Fixed sample shape:", tuple(fixed_images.shape))

In [ ]:
# --- Section 2: Metric Helpers (MSE + SSIM) ---
def denormalize_imagenet(images: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(IMAGENET_MEAN, dtype=images.dtype, device=images.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=images.dtype, device=images.device).view(1, 3, 1, 1)
    return (images * std + mean).clamp(0.0, 1.0)

def ssim_score(predicted: torch.Tensor, target: torch.Tensor, kernel_size: int = 11) -> torch.Tensor:
    padding = kernel_size // 2
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    mu_pred = F.avg_pool2d(predicted, kernel_size, stride=1, padding=padding)
    mu_tgt = F.avg_pool2d(target, kernel_size, stride=1, padding=padding)

    sigma_pred = F.avg_pool2d(predicted * predicted, kernel_size, stride=1, padding=padding) - mu_pred ** 2
    sigma_tgt = F.avg_pool2d(target * target, kernel_size, stride=1, padding=padding) - mu_tgt ** 2
    sigma_cross = F.avg_pool2d(predicted * target, kernel_size, stride=1, padding=padding) - (mu_pred * mu_tgt)

    num = (2 * mu_pred * mu_tgt + c1) * (2 * sigma_cross + c2)
    den = (mu_pred ** 2 + mu_tgt ** 2 + c1) * (sigma_pred + sigma_tgt + c2)
    return (num / (den + 1e-8)).mean()

def batch_mse_ssim(recon_images: torch.Tensor, gt_images: torch.Tensor) -> tuple[float, float]:
    pred = denormalize_imagenet(recon_images)
    tgt = denormalize_imagenet(gt_images)
    mse = F.mse_loss(pred, tgt).item()
    ssim = ssim_score(pred, tgt).item()
    return mse, ssim

In [ ]:
# --- Section 3: Evaluate All Checkpoints and Save Grids ---
checkpoint_paths = sorted(CKPT_DIR.glob("mae_epoch_*.pt"))
best_ckpt = CKPT_DIR / "mae_best.pt"
if best_ckpt.exists() and best_ckpt not in checkpoint_paths:
    checkpoint_paths.append(best_ckpt)

if not checkpoint_paths:
    raise FileNotFoundError(f"No checkpoint files found in {CKPT_DIR}")

print(f"Found {len(checkpoint_paths)} checkpoints")

results = []
for ckpt_path in checkpoint_paths:
    model = load_mae_model(mask_ratio=MASK_RATIO).to(device)
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    with torch.no_grad():
        recon = reconstruct_mae_images(model, fixed_images)

    mse, ssim = batch_mse_ssim(recon, fixed_images)

    masked, _ = apply_patch_mask(fixed_images, mask_ratio=MASK_RATIO)
    n = fixed_images.size(0)
    fig, axes = plt.subplots(3, n, figsize=(4 * n, 9))
    fig.suptitle(f"{ckpt_path.name} | MSE={mse:.5f}, SSIM={ssim:.5f}", fontsize=14)
    for i in range(n):
        axes[0, i].imshow(tensor_to_numpy_image(fixed_images[i]))
        axes[0, i].set_title("Original")
        axes[0, i].axis("off")
        axes[1, i].imshow(tensor_to_numpy_image(masked[i]))
        axes[1, i].set_title("Masked")
        axes[1, i].axis("off")
        axes[2, i].imshow(tensor_to_numpy_image(recon[i]))
        axes[2, i].set_title("Reconstructed")
        axes[2, i].axis("off")

    out_path = COMPARE_DIR / f"{ckpt_path.stem}_compare.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=170, bbox_inches="tight")
    plt.close(fig)

    results.append({
        "checkpoint": ckpt_path.name,
        "mse": float(mse),
        "ssim": float(ssim),
        "image": str(out_path),
    })
    print(f"Done: {ckpt_path.name} | MSE={mse:.5f}, SSIM={ssim:.5f}")

ranked = sorted(results, key=lambda x: (-x["ssim"], x["mse"]))
summary_path = COMPARE_DIR / "checkpoint_ranking.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(ranked, f, indent=2, ensure_ascii=False)

print("\nTop checkpoints by SSIM (higher is better):")
for idx, item in enumerate(ranked[:5], start=1):
    print(f"{idx}. {item['checkpoint']} | SSIM={item['ssim']:.5f} | MSE={item['mse']:.5f}")

print(f"\nSaved ranking to: {summary_path}")
print(f"Saved comparison images to: {COMPARE_DIR}")